# Linear Regression — Hands-on Programming

**Goal.** Implement OLS *from scratch* in NumPy, two ways — closed-form (normal equations) and gradient descent — wrap each in a small scikit-learn-style estimator class, and then cross-check against `sklearn.linear_model.LinearRegression` on synthetic data and a real dataset (the built-in diabetes regression benchmark).

**Role of this notebook.** Implementation and empirical validation. Every formula used here was *derived* in a previous notebook:

| Used here | Where derived |
|---|---|
| Closed form $\theta^* = (X^\top X)^{-1} X^\top y$ | `02_mathematics.ipynb` Theorem 4.2 |
| Gradient $\nabla L(\theta) = (2/n) \cdot X^\top(X \theta - y)$ | `02_mathematics.ipynb` Theorem 3.2 |
| GD update $\theta_{k+1} = \theta_k - \eta \nabla L(\theta_k)$ | `03_optimization.ipynb` §2 |
| Feature scaling rule of thumb | `03_optimization.ipynb` §7 |
| Standard errors, $R^2$, F-test | `04_statistics.ipynb` §§4, 7, 8, 9 |

No new theory in this notebook — only code, sanity checks, and timing.

**Prerequisites.** Notebooks `01_intuition` → `04_statistics` of this folder.

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → `04_statistics` → **`05_hands_on_programming`**.

**Plan.**

1. Imports, seeding, helpers.
2. Synthetic data generator.
3. Implementation A — `LinearRegressionOLS` (closed form via `np.linalg.lstsq` + sanity check vs. `solve`).
4. Implementation B — `LinearRegressionGD` (batch gradient descent, with standardisation).
5. Cross-validation: A vs. B vs. `sklearn.linear_model.LinearRegression` on the same synthetic data.
6. Real data: `sklearn.datasets.load_diabetes()` — fit all three, compare metrics on train / test.
7. Diagnostics: residual plot and QQ-plot to validate the assumptions from `04_statistics.ipynb` §10.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. Synthetic data

A controlled dataset where the true parameters are *known*. The data follows the generative model of `04_statistics.ipynb` (1.1):

> $$y = X \theta_{\text{true}} + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I).$$

We will use it to check that our implementations recover $\theta_{\text{true}}$ within statistical error.

In [ ]:
def make_regression_problem(n=500, p=4, sigma=1.0, seed=0):
    """Generate (X_raw, y) with known true parameters.

    Returns X_raw with NO intercept column (just the features); each estimator
    below adds its own bias column. theta_true is the (p + 1)-vector
    [intercept, slopes...].
    """
    rng_ = np.random.default_rng(seed)
    X_raw = rng_.normal(size=(n, p))
    theta_true = rng_.uniform(-3, 3, size=p + 1)  # +1 for the intercept
    eps = rng_.normal(0, sigma, size=n)
    y = theta_true[0] + X_raw @ theta_true[1:] + eps
    return X_raw, y, theta_true

X_raw, y, theta_true = make_regression_problem(n=500, p=4, sigma=1.0, seed=SEED)
print(f"X_raw shape = {X_raw.shape}")
print(f"y shape     = {y.shape}")
print(f"theta_true  = {theta_true}")

## 2. Implementation A — closed-form OLS

Uses Theorem 4.2 of `02_mathematics.ipynb`: when $\text{rank}(X) = p$,

> $$\theta^* = (X^\top X)^{-1} X^\top y.$$

Per `02_mathematics.ipynb` §4.3, we use `np.linalg.lstsq` rather than forming $(X^\top X)^{-1}$ explicitly — it is more numerically stable (uses an SVD-based driver, no condition-number squaring). We also report the standard errors and $R^2$ from `04_statistics.ipynb`.

In [ ]:
class LinearRegressionOLS:
    """Closed-form OLS via np.linalg.lstsq, sklearn-style API.

    Adds a bias column internally — caller passes raw features X (no 1s column).
    After fit(), attributes match the conventions of 04_statistics.ipynb:
      coef_, intercept_, theta_, sigma2_, se_, t_, p_value_, r2_.
    """

    def fit(self, X, y):
        n, p = X.shape
        X_design = np.column_stack([np.ones(n), X])  # bias trick from 02_mathematics §1.3
        # Solve via lstsq (numerically stable; equivalent to (X^T X)^-1 X^T y at full rank)
        theta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
        self.theta_     = theta
        self.intercept_ = float(theta[0])
        self.coef_      = theta[1:]

        # Residual diagnostics (04_statistics §6, §7, §9)
        y_hat = X_design @ theta
        resid = y - y_hat
        rss   = float(resid @ resid)
        dof   = n - (p + 1)
        self.sigma2_ = rss / dof                                # eq. (6.1)
        XtX_inv     = np.linalg.inv(X_design.T @ X_design)
        self.se_    = np.sqrt(self.sigma2_ * np.diag(XtX_inv))  # eq. (7.1)
        self.t_     = theta / self.se_                          # eq. (8.1)
        self.p_value_ = 2.0 * (1.0 - stats.t.cdf(np.abs(self.t_), df=dof))
        tss     = float(np.sum((y - y.mean()) ** 2))
        self.r2_ = 1.0 - rss / tss                              # eq. (9.2)
        self._n, self._p = n, p + 1
        return self

    def predict(self, X):
        return self.intercept_ + X @ self.coef_

ols = LinearRegressionOLS().fit(X_raw, y)

print(f"theta_true  = {theta_true}")
print(f"theta_hat   = {ols.theta_}")
print(f"max |error| = {np.max(np.abs(ols.theta_ - theta_true)):.4f}")
print(f"R^2         = {ols.r2_:.4f}")
print()
print("Per-coefficient inference:")
print(f"{'j':>2}  {'theta_hat':>10}  {'SE':>8}  {'t':>8}  {'p_value':>10}")
for j in range(ols._p):
    print(f"{j:>2}  {ols.theta_[j]:>10.4f}  {ols.se_[j]:>8.4f}  {ols.t_[j]:>8.2f}  {ols.p_value_[j]:>10.2e}")

**Reading.** With n = 500 and $\sigma$ = 1, all five coefficients land within $\approx$ 0.1 of the truth. The p-values are essentially zero — each feature is overwhelmingly distinguishable from noise. $R^2$ is large because we generated noise-free signal-dominated data.

### 2.1 Sanity check: lstsq agrees with the textbook normal equations

Both code paths solve the *same* mathematical equation (4.1) of `02_mathematics.ipynb`. They should agree to within numerical precision on a well-conditioned problem like this one.

In [ ]:
X_design = np.column_stack([np.ones(len(y)), X_raw])
theta_lstsq = np.linalg.lstsq(X_design, y, rcond=None)[0]
theta_neq   = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)

print(f"theta (lstsq)         = {theta_lstsq}")
print(f"theta (normal eqns)   = {theta_neq}")
print(f"max abs difference    = {np.max(np.abs(theta_lstsq - theta_neq)):.2e}")

## 3. Implementation B — gradient descent

Algorithm 1 from `03_optimization.ipynb` §2.1, with **feature standardisation** (§7.1 of the same notebook) for numerical stability: each non-bias column of X is shifted to mean 0 and scaled to standard deviation 1, then GD is run on the standardised design. The fitted coefficients are mapped back to the original feature scale at the end so that prediction matches Implementation A.

In [ ]:
class LinearRegressionGD:
    """Batch gradient descent for OLS, with internal feature standardisation.

    Hyperparameters chosen so the demo converges in ~200 steps on the
    synthetic data above; tune for your own data.
    """

    def __init__(self, lr=0.1, n_iter=500, tol=1e-8, verbose=False):
        self.lr      = lr
        self.n_iter  = n_iter
        self.tol     = tol
        self.verbose = verbose

    def fit(self, X, y):
        n, p = X.shape
        # Standardise features so the loss is well-conditioned (03_optimization §7.1).
        self.mu_    = X.mean(axis=0)
        self.sigma_ = X.std(axis=0)
        self.sigma_[self.sigma_ == 0] = 1.0  # guard against constant columns
        Xs = (X - self.mu_) / self.sigma_
        Xd = np.column_stack([np.ones(n), Xs])

        theta = np.zeros(p + 1)
        loss_history = []
        for k in range(self.n_iter):
            resid = Xd @ theta - y
            grad  = (2.0 / n) * Xd.T @ resid                  # Theorem 3.2 of 02_math
            theta = theta - self.lr * grad                    # eq. (2.1) of 03_opt
            loss  = float(np.mean(resid ** 2))
            loss_history.append(loss)
            if np.linalg.norm(grad) < self.tol:
                if self.verbose:
                    print(f"Converged at step {k}")
                break

        # Map standardised-feature weights back to original feature scale.
        b_std, w_std = theta[0], theta[1:]
        self.coef_      = w_std / self.sigma_
        self.intercept_ = float(b_std - np.sum(w_std * self.mu_ / self.sigma_))
        self.theta_     = np.r_[self.intercept_, self.coef_]
        self.loss_history_ = np.array(loss_history)
        return self

    def predict(self, X):
        return self.intercept_ + X @ self.coef_

gd = LinearRegressionGD(lr=0.1, n_iter=500).fit(X_raw, y)

print(f"theta_true       = {theta_true}")
print(f"theta_hat (OLS)  = {ols.theta_}")
print(f"theta_hat (GD)   = {gd.theta_}")
print(f"max |GD - OLS|   = {np.max(np.abs(gd.theta_ - ols.theta_)):.2e}")
print(f"final GD loss    = {gd.loss_history_[-1]:.4f}")
print(f"  GD steps used  = {len(gd.loss_history_)}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(gd.loss_history_)
plt.yscale("log")
plt.xlabel("step k")
plt.ylabel("MSE (log scale)")
plt.title("GD loss curve on the synthetic problem")
plt.show()

**Reading.** GD recovers the closed-form solution to roughly six decimal places after a few hundred steps. The loss decreases monotonically (Theorem 4.2 of `03_optimization.ipynb`).

## 4. Cross-validation against scikit-learn

`sklearn.linear_model.LinearRegression` is the reference implementation. Our two estimators should match it within floating-point error on the same data.

In [ ]:
skl = LinearRegression().fit(X_raw, y)
skl_theta = np.r_[skl.intercept_, skl.coef_]

rows = [
    ("truth",          theta_true),
    ("ours: OLS",      ols.theta_),
    ("ours: GD",       gd.theta_),
    ("sklearn",        skl_theta),
]
print(f"{'method':<12}  " + "  ".join(f"{name:>9}" for name in ["b", "w1", "w2", "w3", "w4"]))
for name, vec in rows:
    print(f"{name:<12}  " + "  ".join(f"{v: 9.4f}" for v in vec))

print()
print(f"max |ours OLS  - sklearn| = {np.max(np.abs(ols.theta_  - skl_theta)):.2e}")
print(f"max |ours GD   - sklearn| = {np.max(np.abs(gd.theta_   - skl_theta)):.2e}")

**Reading.** Our closed-form implementation matches sklearn to machine precision (they call the same LAPACK driver). GD matches to within its optimisation tolerance.

## 5. Real data — the diabetes regression benchmark

The diabetes dataset (Efron et al., 2004) ships with scikit-learn. It has 442 patients, 10 numerical features, and a continuous disease-progression target. No download required — it is bundled with the library.

We split into train / test, fit all three estimators on the training half, and compare RMSE and $R^2$ on the held-out test set.

In [ ]:
data = load_diabetes()
X_full, y_full, feature_names = data.data, data.target, data.feature_names
print(f"X shape          = {X_full.shape}")
print(f"y range          = [{y_full.min():.1f}, {y_full.max():.1f}]")
print(f"feature names    = {list(feature_names)}")

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=SEED
)
print(f"\ntrain / test split: {X_train.shape[0]} / {X_test.shape[0]} samples")

In [ ]:
estimators = {
    "ours: OLS":  LinearRegressionOLS(),
    "ours: GD":   LinearRegressionGD(lr=0.01, n_iter=5000),
    "sklearn":    LinearRegression(),
}

results = []
for name, est in estimators.items():
    t0 = time.perf_counter()
    est.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0
    y_pred_tr = est.predict(X_train)
    y_pred_te = est.predict(X_test)
    results.append({
        "name":      name,
        "fit_time":  fit_time,
        "rmse_tr":   float(np.sqrt(mean_squared_error(y_train, y_pred_tr))),
        "rmse_te":   float(np.sqrt(mean_squared_error(y_test,  y_pred_te))),
        "r2_tr":     float(r2_score(y_train, y_pred_tr)),
        "r2_te":     float(r2_score(y_test,  y_pred_te)),
    })

print(f"{'method':<12} {'fit_time(s)':>12} {'RMSE_train':>12} {'RMSE_test':>12} {'R2_train':>9} {'R2_test':>9}")
for r in results:
    print(f"{r['name']:<12} {r['fit_time']:>12.4f} {r['rmse_tr']:>12.2f} {r['rmse_te']:>12.2f} {r['r2_tr']:>9.4f} {r['r2_te']:>9.4f}")

**Reading.** All three estimators land on essentially the same test-set metrics ($R^2$ $\approx$ 0.45 on this benchmark — a known characteristic of the diabetes dataset, not a bug). The closed-form solvers are also faster: GD pays a per-step cost that only pays off when the closed form is infeasible (large p).

**Train > test discrepancy.** The training $R^2$ is higher than the test $R^2$. With a *linear* model and only 10 features on n = 353 training points, overfitting is mild — the gap is small but non-zero. Larger models would show a much wider gap; this is what `00_foundations/04_model_evaluation/` and the regularised methods (Ridge, Lasso) in the next algorithm folders address.

## 6. Residual diagnostics

Two standard plots from `04_statistics.ipynb` §10 to check the Gauss–Markov assumptions:

- **Residual vs. fitted.** If A3 (homoscedasticity) holds, the cloud should be a horizontal band of roughly constant width. A funnel shape would signal heteroscedasticity.
- **QQ-plot of residuals.** If A5 (normality) holds, the points line up on the diagonal. Systematic curvature or fat tails signal a violation.

In [ ]:
ols_full = LinearRegressionOLS().fit(X_train, y_train)
y_pred   = ols_full.predict(X_train)
resid    = y_train - y_pred
resid_std = resid / np.std(resid)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(y_pred, resid, alpha=0.6, edgecolor="k")
axes[0].axhline(0, color="red", lw=1)
axes[0].set_xlabel("fitted value $\\hat y_i$")
axes[0].set_ylabel("residual $r_i$")
axes[0].set_title("Residuals vs. fitted (A3 check)")

stats.probplot(resid_std, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot of standardised residuals (A5 check)")

plt.tight_layout()
plt.show()

print(f"residual mean     = {resid.mean():.4f}   (should be ~0)")
print(f"residual std      = {resid.std():.4f}")
print(f"Shapiro-Wilk p    = {stats.shapiro(resid).pvalue:.4f}   (small => non-normal)")

**Reading.** On diabetes, the residual cloud is a roughly flat band — no strong fan-out, so A3 looks OK. The QQ-plot tracks the diagonal across most of the range with mild tail deviation, and the Shapiro–Wilk p-value (well above 0.05) does not reject normality. So A3 + A5 are plausible on this dataset, and the standard errors / CIs from `04_statistics.ipynb` can be trusted at face value. Per §10 of that notebook: even if A5 were violated, with n = 353 the central limit theorem would still keep CIs asymptotically valid.

## Takeaway

- **Two implementations, one theory.** `LinearRegressionOLS` plugs straight into Theorem 4.2 of `02_mathematics.ipynb`; `LinearRegressionGD` is Algorithm 1 of `03_optimization.ipynb` with the §7.1 feature-scaling fix.
- **Use lstsq, not $(X^\top X)^{-1}$.** Solving the linear system directly avoids squaring the condition number and is the path sklearn (and LAPACK) take.
- **GD $\approx$ closed form.** On well-conditioned data, GD converges to the closed-form $\theta^*$ within numerical tolerance — and to within the same metric as sklearn on real data.
- **Inference for free.** Once we have $\hat{\theta}$ and the residuals, the standard errors, t-statistics, p-values, and $R^2$ from `04_statistics.ipynb` are a handful of extra lines (§2 of this notebook).
- **Diagnostics are cheap and essential.** A residual-vs-fitted plot and a QQ-plot take three lines each and detect the four most common assumption failures.

Next: `../02_polynomial_regression/01_intuition.ipynb`.

**This concludes the Linear Regression folder.** The next algorithm in module 01 is `02_polynomial_regression/`, which keeps the same OLS machinery and only changes the design matrix X to include powers and interaction terms — *linear in parameters, non-linear in features*.